# 🐦 Jar-Qush Tili — TweetyBERT O'qitish (Kaggle T4 GPU)

**Maqsad**: Kanareyka qushining (Serinus canaria) ovoz signallarini TweetyBERT yordamida o'qitish.

## Ishlatish:
1. Kaggle'da yangi notebook yarating
2. **Settings → Accelerator → GPU T4 x2** qo'shing
3. Internet'ni yoqing: **Settings → Internet → On**
4. Run All qiling

## Natijalar:
- `checkpoint.pt` — o'qitilgan model
- Uni Kaggle Output'dan yuklab oling → local M3 da ishlatish uchun

In [ ]:
# ─────────────────────────────────────────────────────────
# 1. MUHIT TEKSHIRUVI
# ─────────────────────────────────────────────────────────
import torch
import subprocess

print('='*50)
print('🔧 MUHIT TEKSHIRUVI')
print('='*50)
print(f'PyTorch versiyasi : {torch.__version__}')
print(f'CUDA mavjud       : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU               : {torch.cuda.get_device_name(0)}')
    print(f'GPU xotirasi      : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# Disk holati
result = subprocess.run(['df', '-h', '/kaggle'], capture_output=True, text=True)
print(f'\nDisk: {result.stdout}')

In [ ]:
# ─────────────────────────────────────────────────────────
# 2. KUTUBXONALAR O'RNATISH
# ─────────────────────────────────────────────────────────
import subprocess

packages = [
    'librosa==0.10.2',
    'soundfile',
    'pydub',
    'tqdm',
    'requests',
    'umap-learn',
    'scipy',
]

print('📦 Kutubxonalar o\'rnatilmoqda...')
for pkg in packages:
    result = subprocess.run(
        ['pip', 'install', '-q', pkg],
        capture_output=True, text=True
    )
    print(f'  ✅ {pkg}')

# ffmpeg (pydub uchun)
subprocess.run(['apt-get', 'install', '-qq', '-y', 'ffmpeg'], capture_output=True)
print('  ✅ ffmpeg')
print('\n✅ Barcha kutubxonalar tayyor!')

In [ ]:
# ─────────────────────────────────────────────────────────
# 3. TWEETYBERT YUKLAB OLISH
# ─────────────────────────────────────────────────────────
import subprocess
from pathlib import Path

if not Path('/kaggle/working/tweety_bert').exists():
    print('🔄 TweetyBERT klonlanmoqda...')
    result = subprocess.run(
        ['git', 'clone', '--quiet', 'https://github.com/georgevenven/tweety_bert.git',
         '/kaggle/working/tweety_bert'],
        capture_output=True, text=True
    )
    print('✅ TweetyBERT klonlandi!')
else:
    print('✅ TweetyBERT allaqachon mavjud')

# Python yo'li qo'shish
import sys
sys.path.insert(0, '/kaggle/working/tweety_bert')
sys.path.insert(0, '/kaggle/working/tweety_bert/src')
print(f'Python path yangilandi: {sys.path[:3]}')

In [ ]:
# ─────────────────────────────────────────────────────────
# 4. XENO-CANTO DATASET YUKLAB OLISH
# ─────────────────────────────────────────────────────────
import requests
import time
import json
from pathlib import Path
from pydub import AudioSegment
from tqdm.notebook import tqdm

RAW_DIR = Path('/kaggle/working/data/raw')
RAW_DIR.mkdir(parents=True, exist_ok=True)

XENO_API = 'https://xeno-canto.org/api/2/recordings'

# ── API so'rovi ──────────────────────────────────────────
query  = 'Serinus canaria q:A type:song'
MAX_N  = 20   # O'qitish uchun ko'proq fayl

print(f'🔍 Xeno-canto qidirilmoqda: {query!r}')
resp = requests.get(XENO_API, params={'query': query}, timeout=30)
resp.raise_for_status()
data = resp.json()

recordings = data.get('recordings', [])[:MAX_N]
print(f'📋 Topildi: {data["numRecordings"]} ta → {len(recordings)} ta yuklanadi')

# ── Yuklab olish ─────────────────────────────────────────
metadata = []
success  = 0

for rec in tqdm(recordings, desc='⬇️ Yuklanmoqda'):
    rec_id   = rec.get('id', 'unknown')
    file_url = rec.get('file', '')
    if not file_url:
        continue
    if not file_url.startswith('http'):
        file_url = 'https:' + file_url

    mp3_path = RAW_DIR / f'xc{rec_id}.mp3'
    wav_path = RAW_DIR / f'xc{rec_id}.wav'

    if wav_path.exists():
        success += 1
        continue

    # MP3 yuklab olish
    try:
        r = requests.get(file_url, timeout=60, stream=True)
        r.raise_for_status()
        with open(mp3_path, 'wb') as f:
            for chunk in r.iter_content(8192):
                f.write(chunk)
    except Exception as e:
        print(f'  ⚠️ {rec_id}: {e}')
        continue

    # MP3 → WAV
    try:
        audio = AudioSegment.from_mp3(mp3_path)
        audio = audio.set_frame_rate(44100).set_channels(1).set_sample_width(2)
        audio.export(wav_path, format='wav')
        mp3_path.unlink(missing_ok=True)
        success += 1
    except Exception as e:
        print(f'  ⚠️ MP3→WAV xato {rec_id}: {e}')
        continue

    metadata.append({
        'id': rec_id, 'file': wav_path.name,
        'country': rec.get('cnt', ''), 'duration': rec.get('length', ''),
        'quality': rec.get('q', ''), 'recordist': rec.get('rec', ''),
    })
    time.sleep(1.0)  # Serverni himoya qilish

# Metadata saqlash
with open(RAW_DIR / 'metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

wav_files = list(RAW_DIR.glob('*.wav'))
print(f'\n✅ Muvaffaqiyatli yuklandi: {success}/{len(recordings)} ta')
print(f'📂 WAV fayllar: {len(wav_files)} ta → {RAW_DIR}')

In [ ]:
# ─────────────────────────────────────────────────────────
# 5. SPEKTROGRAMMA GENERATSIYA (NPZ)
# ─────────────────────────────────────────────────────────
import numpy as np
import librosa
import soundfile as sf
import gc
from pathlib import Path
from scipy.signal import ellip, filtfilt
from tqdm.notebook import tqdm

RAW_DIR  = Path('/kaggle/working/data/raw')
NPZ_DIR  = Path('/kaggle/working/data/processed/npz')
NPZ_DIR.mkdir(parents=True, exist_ok=True)

# TweetyBERT maqolasi parametrlari
NFFT     = 1024
HOP_LEN  = 119
SR_TARGET = 44100  # TweetyBERT 44.1kHz da o'qitilgan

wav_files = sorted(RAW_DIR.glob('*.wav'))
print(f'🎵 {len(wav_files)} ta WAV fayl spektrogrammaga aylantirilmoqda...')

npz_count = 0
for wav_path in tqdm(wav_files, desc='🔄 Spektrogramma'):
    npz_path = NPZ_DIR / f'{wav_path.stem}_segment_0.npz'
    if npz_path.exists():
        npz_count += 1
        continue

    try:
        # Audio o'qish
        data, sr = sf.read(str(wav_path), dtype='int16')
        if data.ndim > 1:
            data = data[:, 0]

        # Minimal uzunlik: 0.5 soniya
        if len(data) / sr < 0.5:
            print(f'  ⚠️ Juda qisqa: {wav_path.name}')
            continue

        # Yuqori o'tkazgich filtr (500 Hz)
        b, a = ellip(5, 0.2, 40, 500 / (sr / 2), 'high')
        data_f = filtfilt(b, a, data.astype(float))

        # STFT
        Sxx    = librosa.stft(data_f, n_fft=NFFT, hop_length=HOP_LEN, window='hann')
        spec_db = librosa.amplitude_to_db(np.abs(Sxx), ref=np.max)  # (freq, time)

        # data_class.py: spectrogram[20:216] → 196 bin
        freq_bins = spec_db.shape[0]
        if freq_bins < 216:
            pad = np.zeros((216 - freq_bins, spec_db.shape[1]))
            spec_db = np.vstack([spec_db, pad])

        spec_cropped = spec_db[20:216, :]  # (196, time)

        # Z-score normalizatsiya
        mean = spec_cropped.mean()
        std  = spec_cropped.std() + 1e-8
        spec_norm = (spec_cropped - mean) / std

        # Vocalization (hamma vaqt birlik) va labels (0 = belgilanmagan)
        time_bins = spec_norm.shape[1]
        vocalization = np.ones(time_bins, dtype=int)
        labels       = np.zeros(time_bins, dtype=int)

        # NPZ saqlash (data_class.py formatida)
        # s: (freq, time) → data_class.py permute(1,0) → (time, freq)
        np.savez(
            str(npz_path),
            s            = spec_norm,       # (196, time)
            vocalization = vocalization,     # (time,)
            labels       = labels,           # (time,)
        )
        npz_count += 1

    except Exception as e:
        print(f'  ❌ Xato {wav_path.name}: {e}')
    finally:
        gc.collect()

print(f'\n✅ NPZ fayllar: {npz_count} ta → {NPZ_DIR}')

In [ ]:
# ─────────────────────────────────────────────────────────
# 6. MODEL VA TRAINER SOZLASH
# ─────────────────────────────────────────────────────────
import sys
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from pathlib import Path

sys.path.insert(0, '/kaggle/working/tweety_bert')
sys.path.insert(0, '/kaggle/working/tweety_bert/src')

from src.model import TweetyBERT
from src.data_class import SongDataSet_Image, CollateFunction

# ── Qurilma ──────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'🖥️  Qurilma: {DEVICE}')
if torch.cuda.is_available():
    print(f'   GPU: {torch.cuda.get_device_name(0)}')

# ── Model parametrlari (maqola defaults) ─────────────────
MODEL_CFG = dict(
    d_transformer      = 256,
    nhead_transformer  = 4,
    num_freq_bins      = 196,
    num_labels         = 40,
    dropout            = 0.1,
    transformer_layers = 3,
    dim_feedforward    = 128,
    m                  = 33,
    p                  = 0.01,
    alpha              = 1,
    length             = 1000,
    pos_enc_type       = 'relative',
    device             = str(DEVICE),
)

model = TweetyBERT(**MODEL_CFG)
model = model.to(DEVICE).float()

total_params = sum(p.numel() for p in model.parameters())
print(f'🧠 Model parametrlari: {total_params:,}')

# ── Dataset va DataLoader ─────────────────────────────────
NPZ_DIR = Path('/kaggle/working/data/processed/npz')
SEGMENT_LEN = 1000  # Vaqt binlarida
BATCH_SIZE  = 8     # T4 xotirasi uchun

dataset = SongDataSet_Image(
    file_dir       = str(NPZ_DIR),
    num_classes    = 40,
    infinite_loader = True,
    segment_length = SEGMENT_LEN,
    pitch_shift    = False,
)

loader = DataLoader(
    dataset,
    batch_size  = BATCH_SIZE,
    num_workers = 2,
    pin_memory  = True,
    collate_fn  = CollateFunction(segment_length=SEGMENT_LEN),
)

# ── Optimizer ─────────────────────────────────────────────
optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=1000, eta_min=1e-6)

print(f'📦 Dataset: {len(list(NPZ_DIR.glob("*.npz")))} ta NPZ fayl')
print(f'🔢 Batch size: {BATCH_SIZE}')
print('✅ Model va Dataset tayyor!')

In [ ]:
# ─────────────────────────────────────────────────────────
# 7. O'QITISH (TRAINING LOOP)
# ─────────────────────────────────────────────────────────
import time
import json
from pathlib import Path
from tqdm.notebook import tqdm

CKPT_DIR    = Path('/kaggle/working/checkpoints')
CKPT_DIR.mkdir(parents=True, exist_ok=True)

# ── O'qitish sozlamalari ──────────────────────────────────
# Kanareyka uchun kichik dataset → kam qadam
# Ko'proq ma'lumot bo'lsa NUM_STEPS ni oshiring
NUM_STEPS   = 2000     # Jami o'qitish qadamlari
LOG_EVERY   = 50       # Har necha qadamda log chiqarish
SAVE_EVERY  = 500      # Har necha qadamda checkpoint saqlash

# ── O'qitish holati ───────────────────────────────────────
model.train()
train_log   = []
best_loss   = float('inf')
data_iter   = iter(loader)

print('🚀 O\'qitish boshlanmoqda...')
print(f'   Jami qadamlar: {NUM_STEPS}')
print(f'   Batch size   : {BATCH_SIZE}')
print(f'   GPU          : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print('─'*50)

start_time = time.time()

for step in tqdm(range(1, NUM_STEPS + 1), desc='🔄 O\'qitish'):
    # ── Batch olish ───────────────────────────────────────
    try:
        batch = next(data_iter)
    except StopIteration:
        data_iter = iter(loader)
        batch = next(data_iter)

    spectrograms, labels, vocalization, _ = batch

    # GPU ga o'tkazish
    spectrograms = spectrograms.to(DEVICE, dtype=torch.float32)
    vocalization = vocalization.to(DEVICE)

    # ── Forward pass ─────────────────────────────────────
    optimizer.zero_grad()

    predictions, mask, original_spec, intermediate = model.train_forward(spectrograms)

    # Loss hisoblash
    loss, masked_loss, unmasked_loss, _ = model.mse_loss(
        predictions     = predictions,
        mask            = mask,
        spec            = original_spec,
        intermediate_layers = intermediate,
        vocalization    = vocalization,
    )

    # ── Backward pass ────────────────────────────────────
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # Gradient clipping
    optimizer.step()
    scheduler.step()

    # ── Loglash ──────────────────────────────────────────
    if step % LOG_EVERY == 0:
        elapsed   = time.time() - start_time
        remaining = elapsed / step * (NUM_STEPS - step)
        lr_now    = scheduler.get_last_lr()[0]

        log_entry = {
            'step'         : step,
            'loss'         : round(loss.item(), 6),
            'masked_loss'  : round(masked_loss.item(), 6),
            'unmasked_loss': round(unmasked_loss.item(), 6),
            'lr'           : round(lr_now, 8),
            'elapsed_min'  : round(elapsed / 60, 2),
        }
        train_log.append(log_entry)

        print(f'  Qadam {step:5d} | loss={loss.item():.5f} | '
              f'masked={masked_loss.item():.5f} | '
              f'lr={lr_now:.2e} | '
              f'qoldi≈{remaining/60:.1f} min')

        # Best model saqlash
        if loss.item() < best_loss:
            best_loss = loss.item()
            torch.save({
                'step'              : step,
                'model_state_dict'  : model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'loss'              : best_loss,
                'model_config'      : MODEL_CFG,
            }, CKPT_DIR / 'best_model.pt')

    # ── Checkpoint saqlash ───────────────────────────────
    if step % SAVE_EVERY == 0:
        ckpt_path = CKPT_DIR / f'checkpoint_step_{step}.pt'
        torch.save({
            'step'              : step,
            'model_state_dict'  : model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'loss'              : loss.item(),
            'model_config'      : MODEL_CFG,
        }, ckpt_path)
        print(f'  💾 Checkpoint saqlandi: {ckpt_path.name}')

# O'qitish logi saqlash
with open(CKPT_DIR / 'train_log.json', 'w') as f:
    json.dump(train_log, f, indent=2)

total_time = (time.time() - start_time) / 60
print(f'\n🎉 O\'qitish tugadi!')
print(f'   Jami vaqt   : {total_time:.1f} daqiqa')
print(f'   Eng yaxshi loss: {best_loss:.6f}')
print(f'   Checkpointlar: {CKPT_DIR}')

In [ ]:
# ─────────────────────────────────────────────────────────
# 8. O'QITISH GRAFIGI
# ─────────────────────────────────────────────────────────
import json
import matplotlib.pyplot as plt
from pathlib import Path

CKPT_DIR = Path('/kaggle/working/checkpoints')

with open(CKPT_DIR / 'train_log.json') as f:
    log = json.load(f)

steps         = [e['step'] for e in log]
losses        = [e['loss'] for e in log]
masked_losses = [e['masked_loss'] for e in log]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Jami loss
axes[0].plot(steps, losses, color='#e74c3c', linewidth=2, label='Jami Loss')
axes[0].set_title('O\'qitish Loss Grafigi', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Qadamlar')
axes[0].set_ylabel('MSE Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)
axes[0].set_facecolor('#f8f9fa')

# Masked loss
axes[1].plot(steps, masked_losses, color='#3498db', linewidth=2, label='Masked Loss')
axes[1].set_title('Masked Region Loss', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Qadamlar')
axes[1].set_ylabel('MSE Loss')
axes[1].legend()
axes[1].grid(alpha=0.3)
axes[1].set_facecolor('#f8f9fa')

plt.suptitle('🐦 Kanareyka TweetyBERT — O\'qitish Jarayoni', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(CKPT_DIR / 'training_plot.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Grafik saqlandi!')

In [ ]:
# ─────────────────────────────────────────────────────────
# 9. CHECKPOINT YUKLAB OLISHGA TAYYORLASH
# ─────────────────────────────────────────────────────────
import shutil
from pathlib import Path

CKPT_DIR = Path('/kaggle/working/checkpoints')
OUTPUT   = Path('/kaggle/working/output')
OUTPUT.mkdir(exist_ok=True)

# Eng yaxshi model nusxasi
shutil.copy(CKPT_DIR / 'best_model.pt',      OUTPUT / 'best_model.pt')
shutil.copy(CKPT_DIR / 'train_log.json',     OUTPUT / 'train_log.json')
shutil.copy(CKPT_DIR / 'training_plot.png',  OUTPUT / 'training_plot.png')

print('📦 Output papkasi tayyor:')
for f in sorted(OUTPUT.iterdir()):
    size_mb = f.stat().st_size / 1e6
    print(f'   {f.name:30s}  {size_mb:.2f} MB')

print('\n✅ Kaggle Output → Download orqali yuklab oling!')
print('📌 Keyingi qadam:')
print('   1. Kaggle → Output → best_model.pt → Download')
print('   2. Local: checkpoints/ papkasiga ko\'chiring')
print('   3. python explore_jar.py --audio data/raw/ --checkpoint checkpoints/best_model.pt')